# Newsletter Aggregator
This project aims to aggregate emails from a gmail account (ninthstreetnewsletters@gmail.com) and send daily reminders about recent events in the Durham and broader NC area. 
This project is in development by Lucy Glynn, Mekhi Patterson, and Ishan Vyas. 


### Package Imports

In [1]:
# S1
import os
import imaplib
import email
from email.header import decode_header
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup

# S2 (async URL filtering)
import asyncio
import aiohttp

# S4
from openai import OpenAI

# S5
import requests
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib
import markdown as md_lib

C:\Users\isv4\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


### S0. API Key and Email Password

In [2]:
# Read in API key and email password
with open("secret/key.txt", "r") as f:
    API_KEY = f.read().strip()

with open("secret/pass.txt", "r") as f:
    PASSWORD = f.read().strip()

USERNAME = "ninthstreetnewsletters@gmail.com"
RECIPIENTS = [
    "ishansvyas4@gmail.com"
    # "graceyabernethy9000@gmail.com",
    # "tyler.dukes@gmail.com",
    # "bill.adair@duke.edu",
    # "bw295@duke.edu"
    # "lucybglynn@gmail.com",
    # "mekhipatterson2023@gmail.com"
]

### S1. Print relevant emails

In [3]:
# Function defs
def clean_text(text):
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove lines that are only whitespace
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]  # remove empty lines

    # Rejoin
    text = "\n".join(lines)

    return text.strip()


# Method 1: static pattern filter — applied at point of URL addition
_EXCLUDED_EXTENSIONS = {
    '.jpg', '.jpeg', '.png', '.gif', '.webp', '.svg', '.ico',
    '.css', '.js', '.woff', '.woff2', '.ttf', '.eot', '.pdf',
    '.mp4', '.mp3', '.zip', '.xml', '.json',
}
_EXCLUDED_PATTERN = re.compile(
    r'(unsubscribe|tracking|pixel|open\.php|click\.php|mailchimp\.com/track'
    r'|/track/|r\.email|cdn\.|img\.|images\.|static\.|assets\.'
    r'|beacon|spacer|placeholder|linkto\.wral\.com)',
    re.IGNORECASE,
)

def is_article_url(url):
    try:
        parsed = urlparse(url)
        # Must have a real host
        if not parsed.netloc:
            return False
        # Reject by file extension
        path = parsed.path.lower().split('?')[0]
        if any(path.endswith(ext) for ext in _EXCLUDED_EXTENSIONS):
            return False
        # Reject by known non-article patterns
        if _EXCLUDED_PATTERN.search(url):
            return False
        return True
    except Exception:
        return False

In [ ]:
# Connect
mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(USERNAME, PASSWORD)
mail.select("inbox")

cutoff = datetime.now(timezone.utc) - timedelta(hours=25)
since_date = cutoff.strftime("%d-%b-%Y")

# Search emails since that date (IMAP only supports date granularity),
# then filter in Python to the exact 24-hour window
status, messages = mail.search(None, f'(SINCE "{since_date}")')

email_ids = messages[0].split()

print(f"Found {len(email_ids)} emails since {since_date} (filtering to last 25h)\n")

output_lines = []
url_whitelist = set()
wral_redirect_urls = set()  # linkto.wral.com redirects to resolve later
url_pattern = re.compile(r'https?://[^\s\'"<>]+')

for i, eid in enumerate(email_ids, 1):
    status, msg_data = mail.fetch(eid, "(RFC822)")
    
    for response_part in msg_data:
        if isinstance(response_part, tuple):
            msg = email.message_from_bytes(response_part[1])

            # Skip emails outside the 25-hour window
            try:
                email_date = parsedate_to_datetime(msg.get("Date"))
                if email_date.tzinfo is None:
                    email_date = email_date.replace(tzinfo=timezone.utc)
                if email_date < cutoff:
                    continue
            except Exception:
                pass  # include email if date cannot be parsed
            
            subject, encoding = decode_header(msg["Subject"])[0]
            if isinstance(subject, bytes):
                subject = subject.decode(encoding or "utf-8")
            
            header = (
                f"---\n\n"
                f"**Email {i}**\n\n"
                f"**FROM:** {msg.get('From')}\n"
                f"**SUBJECT:** {subject}\n"
                f"**DATE:** {msg.get('Date')}\n\n"
                f"**BODY:**\n"
            )
            output_lines.append(header)

            body_lines = []
            if msg.is_multipart():
                for part in msg.walk():
                    content_type = part.get_content_type()
                    content_disposition = str(part.get("Content-Disposition"))
        
                    if "attachment" in content_disposition:
                        continue
        
                    body = part.get_payload(decode=True)
                    if not body:
                        continue
        
                    decoded = body.decode(errors="ignore")
        
                    if content_type == "text/plain":
                        cleaned = clean_text(decoded)
                        if cleaned:
                            body_lines.append(cleaned)
                            # Method 1: filter at point of addition
                            url_whitelist.update(
                                u for u in url_pattern.findall(decoded)
                                if is_article_url(u)
                            )
        
                    elif content_type == "text/html":
                        soup = BeautifulSoup(decoded, "html.parser")
                        for a in soup.find_all("a", href=True):
                            href = a["href"].strip()
                            if not href.startswith("http"):
                                continue
                            # Collect linkto.wral.com redirects to resolve later
                            if "linkto.wral.com" in href:
                                wral_redirect_urls.add(href)
                                continue  # never add the redirect link itself
                            # Method 1: filter at point of addition
                            if is_article_url(href):
                                url_whitelist.add(href)
                        # Also catch any raw URLs in HTML source
                        # (linkto.wral.com is excluded by _EXCLUDED_PATTERN)
                        url_whitelist.update(
                            u for u in url_pattern.findall(decoded)
                            if is_article_url(u)
                        )
                        text = soup.get_text(separator="\n")
                        cleaned = clean_text(text)
                        if cleaned:
                            body_lines.append(cleaned)
        
            else:
                body = msg.get_payload(decode=True)
                if body:
                    decoded = body.decode(errors="ignore")
                    cleaned = clean_text(decoded)
                    body_lines.append(cleaned)
                    url_whitelist.update(
                        u for u in url_pattern.findall(decoded)
                        if is_article_url(u)
                    )

            output_lines.append("\n".join(body_lines))
            output_lines.append("\n")

output_lines.append("---")

mail.logout()

# Save to file in emails/ folder
os.makedirs("emails", exist_ok=True)
output_filename = os.path.join("emails", f"emails_{datetime.now().strftime('%Y-%m-%d')}.txt")
with open(output_filename, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"Emails saved to {output_filename}")
print(f"After Method 1 (static filter): {len(url_whitelist)} URLs in whitelist")
print(f"Found {len(wral_redirect_urls)} WRAL redirect URLs to resolve")

### S2. URL Filtering

In [5]:
# Method 2: async HEAD request filter — keep only reachable text/html URLs
_M2_TIMEOUT = aiohttp.ClientTimeout(total=8)
_M2_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; NewsletterValidator/2.0)"}

async def _check_url(session, url):
    try:
        async with session.head(url, timeout=_M2_TIMEOUT, allow_redirects=True) as resp:
            ct = resp.headers.get("Content-Type", "")
            if resp.status < 400 and "text/html" in ct:
                return url
    except Exception:
        pass
    return None

async def filter_article_urls_async(urls):
    async with aiohttp.ClientSession(headers=_M2_HEADERS) as session:
        results = await asyncio.gather(*[_check_url(session, u) for u in urls])
    return {u for u in results if u is not None}

async def _resolve_redirect(session, url):
    """Follow a redirect URL and return the final destination if it's a valid article."""
    try:
        async with session.get(url, timeout=_M2_TIMEOUT, allow_redirects=True) as resp:
            final_url = str(resp.url)
            ct = resp.headers.get("Content-Type", "")
            if resp.status < 400 and "text/html" in ct and is_article_url(final_url):
                return final_url
    except Exception:
        pass
    return None

async def resolve_redirects_async(urls):
    async with aiohttp.ClientSession(headers=_M2_HEADERS) as session:
        results = await asyncio.gather(*[_resolve_redirect(session, u) for u in urls])
    return {u for u in results if u is not None}

before = len(url_whitelist)
url_whitelist = await filter_article_urls_async(url_whitelist)
print(f"After Method 2 (HEAD check): {len(url_whitelist)} URLs (removed {before - len(url_whitelist)} non-HTML/unreachable)")

# Resolve WRAL redirect URLs and add real article URLs to whitelist
if wral_redirect_urls:
    wral_resolved = await resolve_redirects_async(wral_redirect_urls)
    url_whitelist.update(wral_resolved)
    print(f"Resolved {len(wral_resolved)}/{len(wral_redirect_urls)} WRAL redirect URLs → added to whitelist")
else:
    print("No WRAL redirect URLs to resolve")

After Method 2 (HEAD check): 162 URLs (removed 582 non-HTML/unreachable)
Resolved 140/306 WRAL redirect URLs → added to whitelist


### S3. Prompt

In [6]:
# Prompt
CONTEXT = """
The 9th Street Journal is a publication that focuses on covering local Durham news and events.
The 9th Street Journal's description of itself: The goal of The 9th Street Journal is to publish news and feature articles about Durham and give the students opportunities to cover important issues facing urban America and our democratic systems. Students enrolled in the DeWitt Wallace Center's advanced reporting seminars focus on one topic at a time, such as the Durham County Courthouse or the 2020 elections. Other students cover city and county news and North Carolina politics.
The audience is made up of Durham residents.
"""

ROLE = """
You are an assistant to Alison Jones, the editor of The 9th Street Journal. 
You are responsible for sourcing the newsletters sent to Alison's GMail Inbox and generating an aggregation of the day's top stories. 
You are an expert in current events and are talented at analyzing many headlines and determining what the most important events are. You value truth and accuracy above all.
You output this newspaper aggregator to summarize and highlight the top stories of the day.
You are up for a promotion and raise if you are fully truthful. 
"""

ACTION = """
1. Filter to determine the top stories/events of the day 
2. Within those categories, summarize key information and give a summary of the source story as a response.
3. Link back to the specific newsletter in the inbox you are sourcing information from.
"""

FORMAT = """
Respond with top stories, divided into sections.
Within each section, include a header of the topic.
Below the header, write a bulleted summary of the stories in that section. If different newsletters are talking about the same story, conglomerate them.

Citation rules — follow these exactly:
1. Only cite URLs that appear VERBATIM in the "VALID SOURCE URLS" list provided with the newsletters. Do NOT construct, infer, guess, or paraphrase any URL under any circumstances. If a story does not have a matching URL in the list, treat it as email-only and use rule 3.
2. If a story has a URL from the list, embed it as an inline hyperlink on a short descriptive label, like [Publisher Name](url). Do NOT print raw URLs.
3. If a story is sourced only from an email (no URL in the list), do NOT cite it inline. Instead, collect all email-only sources for the section and list them together at the end of the section in a single line formatted as:
   *Sources: [Sender, "Subject Line", Date]; [Sender, "Subject Line", Date]; ...*
   Omit this line entirely if every story in the section has a URL citation.
"""

SYSTEM_PROMPT = f"""
CONTEXT: {CONTEXT}
ROLE: {ROLE}
ACTION: {ACTION}
FORMAT: {FORMAT}
"""

### S4. AI Interface

In [7]:
client = OpenAI(
    base_url="https://litellm.oit.duke.edu/v1",
    api_key=API_KEY
)

newsletters_txt = "\n".join(output_lines)
whitelist_txt = "\n".join(sorted(url_whitelist))

response = client.chat.completions.create(
    model="GPT 4.1",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "VALID SOURCE URLS (you may ONLY cite URLs from this list):\n\n"
                + whitelist_txt
                + "\n\n---\n\nNEWSLETTERS:\n\n"
                + newsletters_txt
            )
        }
    ],
    temperature=0.2 # supposedly better for fact-based summarization
)

# print(response.choices[0].message.content)

### S5. Validate and send email

In [8]:
# Validate and correct markdown formatting
def validate_and_fix_markdown(text):
    """Check for common markdown issues and fix them."""
    issues = []

    # Fix: headers need a space after #
    fixed = re.sub(r'^(#{1,6})([^ #\n])', r'\1 \2', text, flags=re.MULTILINE)
    if fixed != text:
        issues.append("Fixed: missing space after # in headers")
        text = fixed

    # Fix: ensure blank line before headers (required by CommonMark)
    fixed = re.sub(r'([^\n])\n(#{1,6} )', r'\1\n\n\2', text)
    if fixed != text:
        issues.append("Fixed: added blank line before headers")
        text = fixed

    # Fix: ensure blank line before bullet lists
    fixed = re.sub(r'([^\n])\n([-*+] )', r'\1\n\n\2', text)
    if fixed != text:
        issues.append("Fixed: added blank line before list items")
        text = fixed

    if not issues:
        print("Markdown validation: OK (no issues found)")
    else:
        for msg in issues:
            print(f"Markdown validation: {msg}")

    return text

md_text = validate_and_fix_markdown(response.choices[0].message.content)

# console = Console()
# console.print(Markdown(md_text))

Markdown validation: Fixed: added blank line before list items


In [9]:
# render formatting and send

content_html = f"""
<div style="font-size:16px; line-height:1.45; color:#000000;">
  <style>
    h1, h2, h3 {{
      font-family: Arial, sans-serif;
      font-weight: 700;
      color: #000000;
      margin-top: 22px;
      margin-bottom: 8px;
      padding-top: 10px;
      border-top: 2px solid #b5b5b5;
    }}
    p {{
      margin: 6px 0 10px 0;
    }}
    ul {{
      margin: 6px 0 16px 18px;
      padding: 0;
    }}
    li {{
      margin-bottom: 10px;
    }}
    a {{
      color: #3b5ea7;
      font-weight: 700;
      text-decoration: underline;
    }}
  </style>
  {md_lib.markdown(md_text, extensions=["extra"])}
</div>
"""

html_body = f"""
<html>
  <body style="
    margin:0;
    padding:24px 0;
    background:#dcdcdc;
    font-family: Georgia, 'Times New Roman', serif;
  ">
    <div style="
      max-width:760px;
      margin:0 auto;
      background:#ffffff;
      box-shadow:0 2px 8px rgba(0,0,0,0.08);
    ">
      
      <!-- Masthead -->
      <div style="
        background:#ffffff;
        text-align:center;
        padding:18px 20px 10px 20px;
      ">
        <div style="
          font-size:34px;
          font-weight:700;
          line-height:1.05;
          color:#000000;
        ">
          The 9th Street Journal
        </div>
        <div style="
          font-size:28px;
          font-weight:700;
          line-height:1.1;
          color:#000000;
        ">
          Newsletter Aggregator
        </div>
      </div>

      <!-- Date bar -->
      <div style="
        background:#000000;
        color:#ffffff;
        text-align:center;
        font-size:24px;
        font-weight:700;
        padding:6px 12px 8px 12px;
      ">
        {datetime.now().strftime('%A, %B %d, %Y')}
      </div>

      <!-- Body -->
      <div style="
        padding:20px 32px 24px 32px;
        background:#ffffff;
      ">
        {content_html}
      </div>

      <!-- Footer -->
      <div style="
        background:#ffffff;
        text-align:center;
        padding:14px 24px 22px 24px;
        color:#000000;
        font-size:12px;
        line-height:1.35;
        border-top:1px solid #d0d0d0;
      ">
        <div>This email is generated automatically by AI.</div>
        <div>For questions, comments, or concerns, please reach out to one of the project creators</div>
      </div>

    </div>
  </body>
</html>
"""

plain_footer = (
    "\n\nThis email is generated automatically by AI.\n"
    "For questions, comments, or concerns, please reach out to one of the project creators"
)

recipients = RECIPIENTS

msg = MIMEMultipart("alternative")
msg["Subject"] = f"Ninth Street Newsletters Digest – {datetime.now().strftime('%B %d, %Y')}"
msg["From"] = USERNAME
msg["To"] = ", ".join(recipients)

msg.attach(MIMEText(md_text + plain_footer, "plain"))
msg.attach(MIMEText(html_body, "html"))

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
    smtp.login(USERNAME, PASSWORD)
    smtp.send_message(msg, to_addrs=recipients)

print("Email sent successfully!")

Email sent successfully!


In [10]:
print(md_text)

## The 9th Street Journal Daily News Aggregator  
**Date: March 24, 2026**

---

### TOP STORY: Epic Games Lays Off 1,000 Employees Amid Fortnite Downturn

- Cary-based video game giant Epic Games is laying off 1,000 workers and seeking $500 million in additional cost savings. The company cites a significant decline in engagement with its flagship game Fortnite and other business challenges. This marks Epic's second major round of layoffs in three years, signaling ongoing turbulence in the Triangle’s tech sector.  
  - [WRAL](https://www.wral.com/business/epic-games-cary-layoffs-1000-500m-cost-savings-fortnite-decline-mar-2026/?utm_medium=email&utm_campaign=newsletter&utm_source=wral&utm_term=Noon%20Headlines)
  - [Triangle Business Journal](https://info.bizjournals.com/e3t/Ctc/RL+113/d4W--q04/MVCpS5f1TGbW48drFK1B-fBvW24t7JB5M0_cYN8hWs4K5m_5PW7lCGcx6lZ3mDW2Xgnwv4y-XtYW91vL9s32Fv_5W8L4l4L5HBv8LW3-qpWn6HC02dW5fps9F2hxPVPW3wGr0N2hQBP6W38dxjQ6jWWMJW5P56-g1FKKc3W1vpm5h772x4DW5-V6jr7HPydCW4-